# Get WhisperTimeSync

In [1]:
# !rm -rf WhisperTimeSync
# !git clone https://github.com/EtienneAb3d/WhisperTimeSync.git

In [2]:
# !apt install -y ffmpeg
# !pip install scipy soundfile tqdm six torch transformers vad librosa srt numpy==2.0

# Transcribe

In [3]:
import json
from get_torah_text_using_sefaria import get_chapter_string
import scipy.io.wavfile as wavfile
import io
from six.moves.urllib.request import urlopen
import pathlib
import soundfile as sf
from tqdm import tqdm
from nikud_and_teamim import remove_nikud, replace_teamim_with_emphasis, remove_nikud_and_teamim
from remove_nikud_dicta import remove_nikud_dicta

import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import librosa
import srt
from datetime import timedelta
from transformers import pipeline
import os
import soundfile as sf
import scipy.signal
from vad import EnergyVAD 
from tqdm import tqdm




!nvidia-smi


/home/prj8045/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Thu Dec  5 18:33:22 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.107.02             Driver Version: 550.107.02     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               Off |   00000000:01:00.0 Off |                  Off |
| 31%   23C    P8             17W /  230W |   20145MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# choose the GPU with empty memory
mem = !nvidia-smi --query-gpu=memory.free --format=csv
mem = mem[1:]
mem = [int(m.replace(' MiB', '')) for m in mem]
device = torch.device(f'cuda:{mem.index(max(mem))}' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda', index=1)

we want to adapt the text to time, in iterations.
1) Ketiv Maleh (בֹּקֶר -> בוקר) - becuse we use hebrew model that trained on modern hebrew in without nikud in Ketiv Male this is the first step to adapt the text to the model.
2) Ketiv Haser without teamim - it closer to our data.
3) Ketiv Haser with teamim - now it is much simpler to adapt the text to the model.

In [5]:
# get the text
with open("text.txt", "r") as f:
    # first lines
    original_text = f.read()

# prepare the text
original_text = original_text.replace("׃ \n", "׃ ").replace("־", "־ ")


# the first steps are ketiv maleh, for that we use the API of Dicta
ketiv_maleh = remove_nikud_dicta(original_text) # step 2

# before the step of ketiv maleh we want another step of text without "׃" or "־".
cleaned_ketiv_maleh = ketiv_maleh.replace("־", "").replace("׃", ".") # step 1

# the second step is the ketiv haser without teamim, for that we use our own library
ketiv_haser_no_teamim = remove_nikud_and_teamim(original_text) # step 3

# the third step is the ketiv haser with teamim, for that we use our own library
ketiv_haser_teamim = remove_nikud(original_text) # step 4



# save all the steps
with open("step01.txt", "w") as f:
    f.write(cleaned_ketiv_maleh)
with open("step02.txt", "w") as f:
    f.write(ketiv_maleh)
with open("step03.txt", "w") as f:
    f.write(ketiv_haser_no_teamim)
with open("step04.txt", "w") as f:
    f.write(ketiv_haser_teamim)
with open("final_step.txt", "w") as f:
    f.write(original_text)

print("step 1: \n", cleaned_ketiv_maleh)
print("step 2: \n", ketiv_maleh)
print("step 3: \n", ketiv_haser_no_teamim)
print("step 4: \n", ketiv_haser_teamim)
print("with nikud: \n", original_text.split("\n")[0])

step 1: 
 וזאת הברכה אשר בירך משה איש האלוהים את בני ישראל לפני מותו. ויאמר יהוה מסיני בא וזרח משעיר למו הופיע מהר פארן ואתה מרבבת קודש מימינו למו. אף חבב עמים כל קדשיו בידך והם תוכו לרגלך יישא מדברתיך. תורה ציוה לנו משה מורשה קהילת יעקוב. ויהי בישורון מלך בהתאסף ראשי עם יחד שבטי ישראל. יחי ראובן ואל ימת ויהי מתיו מספר. וזאת ליהודה ויאמר שמע יהוה קול יהודה ואל עמו תביאנו ידיו רב לו ועזר מצריו תהיה. וללוי אמר תומיך ואוריך לאיש חסידך אשר ניסיתו במסה תריביהו על מי מריבה. האמר לאביו ולאימו לא ראיתיו ואת אחיו לא הכיר ואת בנו לא ידע כי שמרו אמרתך ובריתך ינצרו. יורו משפטיך ליעקב ותורתך לישראל ישימו קטורה באפך וכליל על מזבחך. ברך יהוה חילו ופועל ידיו תרצה מחץ מותניים קמיו ומשנאיו מן יקומון. לבנימן אמר ידיד יהוה ישכון לבטח עליו חפף עליו כל היום ובין כתפיו שכן. וליוסף אמר מבורכת יהוה ארצו ממגד שמיים מיטל ומתהום רבצת תחת. וממגד תבואת שמש וממגד גרש ירחים. ומראש הררי קדם וממגד גבעות עולם. וממגד ארץ ומלאה ורצון שכני סנה תבואתה לראש יוסף ולקודקוד נזיר אחיו. בכור שורו הדר לו וקרני ראם קרניו בהם עמים י

In [6]:
# Load the fine-tuned model and processor
# model = WhisperForConditionalGeneration.from_pretrained("ivrit-ai/whisper-v2-pd1-e1").to(device)
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v2").to(device)
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v2")

model.generation_config.language = "he"

# Define maximum audio segment length
MAX_SEGMENT_LENGTH = 30 * 1000  # 30 seconds in milliseconds



In [7]:
WITH_TIMESTAMPS = False # True, False, or "word". True will use the first method, "word" is the best method for timestamps in whisper
MAX_SEGMENT_LENGTH_CHARS = 9999 # maximum number of characters in each segment in the SRT file 

In [24]:

# Function to split audio file into segments
def split_audio(audio_file, output_dir):
    """
    Args:
        audio_file: Path to the input audio file.
        output_dir: Directory to save the temporary audio segments.

    Returns:
        A list of file names for the created audio segments.
    """
    FRAME_LENGTH = 20  # in milliseconds
    vad = EnergyVAD(
        sample_rate=16000,
        frame_length=FRAME_LENGTH,
        frame_shift=FRAME_LENGTH,
        energy_threshold=0.002,
        pre_emphasis=0.95,
    )

    audio, sr = librosa.load(audio_file, sr=16000)
    voice_activity = vad(audio)

    # Apply median filter to smooth the voice activity detection
    voice_activity_median = scipy.signal.medfilt(voice_activity, kernel_size=15)
    segments = []
    segment_files = []
    start = 0
    total_frames = len(voice_activity_median)

    base_filename = os.path.splitext(os.path.basename(audio_file))[0]

    while start < total_frames:
        for end in range(min(start + MAX_SEGMENT_LENGTH // FRAME_LENGTH - 1, total_frames), start, -1):
            if end >= len(voice_activity_median):
                end = len(voice_activity_median) - 1
            if not voice_activity_median[end]:
                break

        segment = audio[start * FRAME_LENGTH * sr // 1000:end * FRAME_LENGTH * sr // 1000]
        segments.append(segment)

        segment_filename = f"{base_filename}_{len(segments):03d}.wav"
        segment_path = os.path.join(output_dir, segment_filename)
        sf.write(segment_path, segment, sr)
        segment_files.append(segment_filename)

        start = end + 1
    # if the last one is too short (less than 0.1 seconds), we throw it away
    if len(segments) > 1:
        last_segment_duration = len(segments[-1]) / sr
        if last_segment_duration < 0.1:
            os.remove(os.path.join(output_dir, segment_files[-1]))
            segment_files.pop()
            segments.pop()
    
    return segment_files
    



def create_srt_segment(result, start_time=0, last_index=0):
    """
    Creates a list of Subtitle objects from the Whisper result.

    Args:
        result: The output from the Whisper pipeline.
        start_time: The starting time (in seconds) for this segment within the full audio.
        last_index: The last subtitle index from the previous segment.

    Returns:
        A list of Subtitle objects and the last index used.
    """
    subtitles = []
    current_line = ""
    segment_start_time = None
    current_index = last_index + 1

    for i, segment in enumerate(result["chunks"]):
        word = segment["text"].strip()
        if segment_start_time is None:
            segment_start_time = segment["timestamp"][0]

        if len(current_line + word) > MAX_SEGMENT_LENGTH_CHARS:
            end_time = segment["timestamp"][0] + start_time
            subtitles.append(srt.Subtitle(index=current_index,
                                          start=timedelta(seconds=segment_start_time + start_time),
                                          end=timedelta(seconds=end_time),
                                          content=current_line.strip()))
            current_index += 1
            current_line = word + " "
            segment_start_time = segment["timestamp"][0]
        else:
            current_line += word + " "

    # Add the last subtitle if there's remaining content
    if current_line:
        subtitles.append(srt.Subtitle(index=current_index,
                                      start=timedelta(seconds=segment_start_time + start_time),
                                      end=timedelta(seconds=result["chunks"][-1]["timestamp"][1] + start_time),
                                      content=current_line.strip()))
        current_index += 1

    return subtitles, current_index - 1

def process_audio_segment(audio_segment, start_time=0, last_index=0):
    """
    Processes a single audio segment to generate the SRT content.

    Args:
        audio_segment: The audio data for this segment.
        start_time: The starting time (in seconds) for this segment within the full audio.
        last_index: The last subtitle index from the previous segment.

    Returns:
        A list of Subtitle objects for the given segment and the last index used.
    """
    asr = pipeline("automatic-speech-recognition", model=model, tokenizer=processor.tokenizer,
                   feature_extractor=processor.feature_extractor, device=device)

    result = asr(audio_segment, return_timestamps=WITH_TIMESTAMPS,
                 generate_kwargs={"language": "<|he|>",
                                  "task": "transcribe"})
    if not WITH_TIMESTAMPS:
        result["chunks"] = [{'text': result["text"], 'timestamp': (0, (len(audio_segment) // 160)/100)}] # timestamp is in seconds with 2 decimal places
    subtitles, last_index = create_srt_segment(result, start_time, last_index)
    return subtitles, last_index

def generate_srt_from_audio(audio_file, output_srt_file):
    """
    Generates an SRT file from an audio file.

    Args:
        audio_file: Path to the input audio file.
        output_srt_file: Path to the output SRT file.
    """
    output_dir = "temp_audio_segments"
    os.makedirs(output_dir, exist_ok=True)

    segment_files = split_audio(audio_file, output_dir)

    all_subtitles = []
    total_duration = 0
    last_index = 0

    for i, segment_file in enumerate(tqdm(segment_files)):
        segment_path = os.path.join(output_dir, segment_file)
        audio_segment, _ = librosa.load(segment_path, sr=16000)
        segment_subtitles, last_index = process_audio_segment(audio_segment, total_duration, last_index)
        all_subtitles.extend(segment_subtitles)
        total_duration += librosa.get_duration(y=audio_segment, sr=16000)  # in seconds
    
    # Compose the final SRT content from all subtitles
    full_srt_content = srt.compose(all_subtitles)
    
    with open(output_srt_file, "w", encoding="utf-8") as f:
        f.write(full_srt_content)

    # Remove temporary audio segments
    for segment_file in segment_files:
        os.remove(os.path.join(output_dir, segment_file))
    os.rmdir(output_dir)

<div dir="rtl">
<h1>
גרסה ממוקבלת
</h1>
מהירה פי 3, הזמנים לא עקביים עם מה שהיה עד עכשיו, אז צריך עוד לתקן את זה
</div>

In [29]:
# def process_audio_batch(audio_segments, start_times, last_index=0):
#     """
#     Processes a batch of audio segments to generate SRT content.

#     Args:
#         audio_segments: List of audio segments to process
#         start_times: List of start times for each segment
#         last_index: Last subtitle index from previous batch

#     Returns:
#         List of subtitles and the last index used
#     """
#     batch_size = len(audio_segments)
#     print(f"Processing batch of {batch_size} segments")
#     asr = pipeline("automatic-speech-recognition", model=model, tokenizer=processor.tokenizer,
#                    feature_extractor=processor.feature_extractor, device=device, batch_size=len(audio_segments))

#     # Process batch
#     results = asr(audio_segments, return_timestamps=WITH_TIMESTAMPS,
#                  generate_kwargs={"language": "<|he|>",
#                                 "task": "transcribe"})
    
#     all_subtitles = []
#     current_index = last_index
    
#     # Process each result in the batch
#     for result, start_time in zip(results, start_times):
#         if not WITH_TIMESTAMPS:
#             result["chunks"] = [{'text': result["text"], 
#                                'timestamp': (0, (len(audio_segments[0]) // 160)/100)}]
#         subtitles, current_index = create_srt_segment(result, start_time, current_index)
#         all_subtitles.extend(subtitles)
    
#     return all_subtitles, current_index

# def create_srt_segment(result, start_time=0, last_index=0):
#     subtitles = []
#     current_line = ""
#     segment_start_time = None
#     current_index = last_index + 1

#     for i, segment in enumerate(result["chunks"]):
#         word = segment["text"].strip()
#         if segment_start_time is None:
#             segment_start_time = segment["timestamp"][0]

#         if len(current_line + word) > MAX_SEGMENT_LENGTH_CHARS:
#             # Fix: Use exact timestamp for end time
#             end_time = segment["timestamp"][0] + start_time
#             subtitles.append(srt.Subtitle(
#                 index=current_index,
#                 start=timedelta(seconds=segment_start_time + start_time),
#                 end=timedelta(seconds=end_time),
#                 content=current_line.strip()
#             ))
#             current_index += 1
#             current_line = word + " "
#             segment_start_time = segment["timestamp"][0]
#         else:
#             current_line += word + " "

#     # Add the last subtitle with correct end time
#     if current_line:
#         end_time = result["chunks"][-1]["timestamp"][1] + start_time
#         subtitles.append(srt.Subtitle(
#             index=current_index,
#             start=timedelta(seconds=segment_start_time + start_time),
#             end=timedelta(seconds=end_time),
#             content=current_line.strip()
#         ))
#         current_index += 1

#     # Validate and adjust overlapping timestamps
#     for i in range(len(subtitles) - 1):
#         if subtitles[i].end > subtitles[i + 1].start:
#             subtitles[i].end = subtitles[i + 1].start

#     return subtitles, current_index - 1

# def generate_srt_from_audio(audio_file, output_srt_file, batch_size=2):
#     output_dir = "temp_audio_segments"
#     os.makedirs(output_dir, exist_ok=True)

#     # Add progress bar for audio splitting
#     print("Splitting audio into segments...")
#     segment_files = split_audio(audio_file, output_dir)

#     all_subtitles = []
#     total_duration = 0
#     last_index = 0

#     # Add progress bar for processing
#     print("Processing audio segments...")
#     with tqdm(total=len(segment_files), desc="Processing segments") as pbar:
#         for i in range(0, len(segment_files), batch_size):
#             batch_files = segment_files[i:i + batch_size]
#             batch_segments = []
#             start_times = []
            
#             for segment_file in batch_files:
#                 segment_path = os.path.join(output_dir, segment_file)
#                 audio_segment, _ = librosa.load(segment_path, sr=16000)
#                 batch_segments.append(audio_segment)
#                 start_times.append(total_duration)
#                 total_duration += librosa.get_duration(y=audio_segment, sr=16000)

#             batch_subtitles, last_index = process_audio_batch(batch_segments, start_times, last_index)
#             all_subtitles.extend(batch_subtitles)
#             pbar.update(len(batch_files))

#     print("Generating SRT file...")
#     full_srt_content = srt.compose(all_subtitles)
    
#     with open(output_srt_file, "w", encoding="utf-8") as f:
#         f.write(full_srt_content)

#     # Cleanup with progress indication
#     print("Cleaning up temporary files...")
#     for segment_file in segment_files:
#         os.remove(os.path.join(output_dir, segment_file))
#     os.rmdir(output_dir)
#     print("Done!")

In [ ]:
dir = "/home/prj8045/Torah-reading-data--alignment-and-slicer/automatic using WhisperTimeSync/"
# Call the main function
generate_srt_from_audio(dir + "audio.mp3", dir + "output.srt")

Splitting audio into segments...
Processing audio segments...


Processing segments:   0%|          | 0/22 [00:00<?, ?it/s]

Processing batch of 2 segments


Processing segments:   9%|▉         | 2/22 [00:01<00:19,  1.00it/s]

Processing batch of 2 segments


...

Processing batch of 2 segments


Processing segments:  91%|█████████ | 20/22 [00:23<00:02,  1.16s/it]

Processing batch of 2 segments


Processing segments: 100%|██████████| 22/22 [00:25<00:00,  1.15s/it]

Generating SRT file...
Cleaning up temporary files...
Done!


In [11]:
# !apt install -y java

In [12]:
# # Now we have the srt file, with time but the low quality text, and the text file with the high quality text.
# # We will use the WhisperTimeSync to sync the two files, and get the srt file with the high quality text.
# !java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "output.srt" "text.txt" he


In [13]:
# We will run the WhisperTimeSync on the output and the step files
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "output.srt" "step01.txt" he
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "step01.txt.srt" "step02.txt" he
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "step02.txt.srt" "step03.txt" he
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "step03.txt.srt" "step04.txt" he
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "step04.txt.srt" "final_step.txt" he



Output (step01.txt.srt):

Output (step02.txt.srt):

Output (step03.txt.srt):

Output (step04.txt.srt):

Output (final_step.txt.srt):


In [14]:
# try do it in one step
# !java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "output.srt" "final_step.txt" he

# The method for Milra and Milael future project:

In [15]:
# WITH_TEAMIM = False
# WITH_STRESS = True
# SR = 16000


# if WITH_TEAMIM and WITH_STRESS:
#     print("WITH_TEAMIM and WITH_STRESS cannot be True at the same time.")
#     exit(1)


In [16]:
# Example with the first chapter of Genesis:
with open('links_for_audio_from_929.json', 'r') as f:
    links_for_audio_from_929 = json.load(f)

url = links_for_audio_from_929["books"][0]["chapters"][0]["link"]

# Download the audio file
z = io.BytesIO(urlopen(url).read())
pathlib.Path((f"{book_name}_{chapter}.mp3")).write_bytes(z.getbuffer())
audio, sr = librosa.load(f"{book_name}_{chapter}.mp3", sr=SR)


# Get the text from Sefaria
book_name = "Genesis"
chapter = 1
text = get_chapter_string(book_name, chapter)

# remove the nikud and replace the teamim with empsis
if WITH_TEAMIM:
    text = remove_nikud(text)
else:
    if WITH_STRESS:
        text = remove_nikud(text)
        text = replace_teamim_with_emphasis(text)
    else:
        text = remove_nikud_and_teamim(text)
text = "בראשֽית " + "פרק " + "אלף " + text

with open(f"{book_name}_{chapter}.txt", "w") as f:
    f.write(text)

# Transcribe the audio to get the timestamps (as srt file)
!python3 WhisperTimeSync/transcribe.py /content/"{book_name}_{chapter}.mp3" large-v3

# Align(sync) the text with the audio
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "{book_name}_{chapter}.srt" "{book_name}_{chapter}.txt" he

# Load the srt file
with open(f"{book_name}_{chapter}.txt.srt", "r") as f:
    srt = f.read()

dataset = {"text": [], "start": [], "end": [], "audio_file": []}
# Add the times, text, and audio file name(save each sentence as a separate audio file), to the dataset
for i, line in enumerate(srt.split("\n")):
    if i % 4 == 0:
        dataset["start"].append(line)
    elif i % 4 == 1:
        dataset["end"].append(line)
    elif i % 4 == 2:
        dataset["text"].append(line)
    elif i % 4 == 3:
        dataset["audio_file"].append(f"{book_name}_{chapter}_{i//4}.wav")
        start = int(float(dataset["start"][-1].replace(",", ".").replace(" --> ", "")) * SR)
        end = int(float(dataset["end"][-1].replace(",", ".").replace(" --> ", "")) * SR)
        sf.write(f"{book_name}_{chapter}_{i//4}.wav", audio[start:end], SR)


FileNotFoundError: [Errno 2] No such file or directory: 'links_for_audio_from_929.json'

In [ ]:
import librosa
import requests
import json
from get_torah_text_using_sefaria import get_chapter_string
import scipy.io.wavfile as wavfile
import io
from six.moves.urllib.request import urlopen
import pathlib
import soundfile as sf
from tqdm import tqdm
SR = 16000


books_data = [
    {"name": "Bereshit", "number": 1, "chapters": 50, "sefaria_name": "Genesis"},
    {"name": "Shemot", "number": 2, "chapters": 40, "sefaria_name": "Exodus"},
    {"name": "Vaikra", "number": 3, "chapters": 27, "sefaria_name": "Leviticus"},
    {"name": "Bamidbar", "number": 4, "chapters": 36, "sefaria_name": "Numbers"},
    {"name": "Dvarim", "number": 5, "chapters": 34, "sefaria_name": "Deuteronomy"}
]
sefer_names = ["", "בְּרֵאשִֽׁית", "שְׁמֽות", "וַיִּקְרָֽא", "בַּמִּדְבָּֽר", "דְּבָרִֽים"]
otiyot_gematria = ["", "אָֽלֶף", "בֵּֽית", "גִּֽימֶל", "דָּֽלֶת", "הֵֽא", "וָֽו", "זַֽיִן", "חֵֽית", "טֵֽית", "יֽוֹד", "יֽוֹד-אָֽלֶף", "יֽוֹד-בֵּֽית", "יֽוֹד-גִּֽימֶל", "יֽוֹד-דָּֽלֶת", "טֵֽית-וָֽו", "טֵֽית-זַֽיִן", "יֽ-זַֽיִן", "יֽוֹד-חֵֽית", "יֽוֹד-טֵֽית", "כַּֽף", "כַּֽף-אָֽלֶף", "כַּֽף-בֵּֽית", "כַּֽף-גִּֽימֶל", "כַּֽף-דָּֽלֶת", "כַּֽף-הֵֽא", "כַּֽף-וָֽו", "כַּֽף-זַֽיִן", "כַּֽף-חֵֽית", "כַּֽף-טֵֽית", "לָֽמֶד", "לָֽמֶד-אָֽלֶף", "לָֽמֶד-בֵּֽית", "לָֽמֶד-גִּֽימֶל", "לָֽמֶד-דָּֽלֶת", "לָֽמֶד-הֵֽא", "לָֽמֶד-וָֽו", "לָֽמֶד-זַֽיִן", "לָֽמֶד-חֵֽית", "לָֽמֶד-טֵֽית", "מֵֽם", "מֵֽם-אָֽלֶף", "מֵֽם-בֵּֽית", "מֵֽם-גִּֽימֶל", "מֵֽם-דָּֽלֶת", "מֵֽם-הֵֽא", "מֵֽם-וָֽו", "מֵֽם-זַֽיִן", "מֵֽם-חֵֽית", "מֵֽם-טֵֽית", "נֽוּן", "נֽוּן-אָֽלֶף", "נֽוּן-בֵּֽית", "נֽוּן-גִּֽימֶל", "נֽוּן-דָּֽלֶת", "נֽוּן-הֵֽא", "נֽוּן-וָֽו", "נֽוּן-זַֽיִן", "נֽוּן-חֵֽית", "נֽוּן-טֵֽית", "סָֽמֶךְ"]
otiyot_gematria = [remove_nikud(ot) for ot in otiyot_gematria]
dataset = {"text": [], "audio_file": []}

with open('links_for_audio_from_929.json', 'r') as f:
    links_for_audio_from_929 = json.load(f)

# Calculate total number of chapters
total_chapters = sum(book["chapters"] for book in books_data)

# Create a progress bar
pbar = tqdm(total=total_chapters)

# Loop over all the chapters of all the books
for book in books_data:
    book_name = book["name"]
    sefaria_name = book["sefaria_name"]
    for chapter in range(1, book["chapters"]+1):
        if chapter % 27 != 0: # test on small data
            continue
        url = links_for_audio_from_929["books"][book["number"]-1]["chapters"][chapter-1]["link"]
        z = io.BytesIO(urlopen(url).read())
        pathlib.Path((f"{book_name}_{chapter}.mp3")).write_bytes(z.getbuffer())
        audio, sr = librosa.load(f"{book_name}_{chapter}.mp3", sr=SR)
        text = get_chapter_string(sefaria_name, chapter)
        if WITH_TEAMIM:
            text = remove_nikud(text)
        else:
            if WITH_STRESS:
                text = remove_nikud(text)
                text = replace_teamim_with_emphasis(text)
            else:
                text = remove_nikud_and_teamim(text)
        text = f"{sefer_names[book['number']]} " + f"פרק {otiyot_gematria[chapter]} " + text
        with open(f"{book_name}_{chapter}.txt", "w") as f:
            f.write(text)
        !python3 WhisperTimeSync/transcribe.py /content/"{book_name}_{chapter}.mp3" large-v3
        !java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "{book_name}_{chapter}.srt" "{book_name}_{chapter}.txt" he
        with open(f"{book_name}_{chapter}.txt.srt", "r") as f:
            srt = f.read()

        for i, line in enumerate(srt.split("\n")):
            if i % 4 == 0: # The line is a number of the subtitle
                continue
            if i % 4 == 1: # The line is a time
                start, end = line.split(" --> ")
                start = start.replace(",", ".")
                end = end.replace(",", ".")
                start = start.split(":")
                end = end.split(":")
                start = float(start[0])*3600 + float(start[1])*60 + float(start[2])
                end = float(end[0])*3600 + float(end[1])*60 + float(end[2])
                start = float(start)
                end = float(end)
                continue
            if i % 4 == 2: # The line is the text
                dataset["text"].append(line)
                dataset["audio_file"].append(f"{book_name}_{chapter}_{int(i/4):02}.wav")
                # Save the audio file
                sf.write(f"{book_name}_{chapter}_{int(i/4):02}.wav", audio[int(start*SR):int(end*SR)], SR)
            if i % 4 == 3: # The line is an empty line
                continue
        # Remove the original audio file and the srt file
        !rm "{book_name}_{chapter}.mp3"
        !rm "{book_name}_{chapter}.txt"
        !rm "{book_name}_{chapter}.txt.srt"
        !rm "{book_name}_{chapter}.srt"

        # Update the progress bar
        pbar.update()

# Save the dataset
with open("dataset.json", "w") as f:
    json.dump(dataset, f)
print("Done")

In [ ]:

book_name = "Bereshit"
chapter = 27
url = links_for_audio_from_929["books"][1]["chapters"][chapter-1]["link"]
z = io.BytesIO(urlopen(url).read())
pathlib.Path((f"{book_name}_{chapter}.mp3")).write_bytes(z.getbuffer())
!python3 WhisperTimeSync/transcribe.py /content/"{book_name}_{chapter}.mp3" large-v3

In [ ]:
# Save the dataset
with open("dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f)

In [ ]:
# play random audio and print its text
import random
import IPython.display as ipd
index = random.randint(0, len(dataset["audio_file"]))
print(dataset["text"][index])
ipd.Audio(dataset["audio_file"][index])


In [ ]:
index = 14
print(dataset["text"][index])
ipd.Audio(dataset["audio_file"][index])

In [ ]:
# Sync the times of the real text using the srt file
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "{book_name}_{chapter}.srt" "{book_name}_{chapter}.txt" he

In [ ]:
books_data = [
    {"name": "Bereshit", "number": 1, "chapters": 50, "sefaria_name": "Genesis"},
    {"name": "Shemot", "number": 2, "chapters": 40, "sefaria_name": "Exodus"},
    {"name": "Vaikra", "number": 3, "chapters": 27, "sefaria_name": "Leviticus"},
    {"name": "Bamidbar", "number": 4, "chapters": 36, "sefaria_name": "Numbers"},
    {"name": "Dvarim", "number": 5, "chapters": 34, "sefaria_name": "Deuteronomy"}
]
sefer_names = ["", "בְּרֵאשִֽׁית", "שְׁמֽות", "וַיִּקְרָֽא", "בַּמִּדְבָּֽר", "דְּבָרִֽים"]
otiyot_gematria = ["", "אָֽלֶף", "בֵּֽית", "גִּֽימֶל", "דָּֽלֶת", "הֵֽא", "וָֽו", "זַֽיִן", "חֵֽית", "טֵֽית", "יֽוֹד", "יֽוֹד-אָֽלֶף", "יֽוֹד-בֵּֽית", "יֽוֹד-גִּֽימֶל", "יֽוֹד-דָּֽלֶת", "טֵֽית-וָֽו", "טֵֽית-זַֽיִן", "יֽ-זַֽיִן", "יֽוֹד-חֵֽית", "יֽוֹד-טֵֽית", "כַּֽף", "כַּֽף-אָֽלֶף", "כַּֽף-בֵּֽית", "כַּֽף-גִּֽימֶל", "כַּֽף-דָּֽלֶת", "כַּֽף-הֵֽא", "כַּֽף-וָֽו", "כַּֽף-זַֽיִן", "כַּֽף-חֵֽית", "כַּֽף-טֵֽית", "לָֽמֶד", "לָֽמֶד-אָֽלֶף", "לָֽמֶד-בֵּֽית", "לָֽמֶד-גִּֽימֶל", "לָֽמֶד-דָּֽלֶת", "לָֽמֶד-הֵֽא", "לָֽמֶד-וָֽו", "לָֽמֶד-זַֽיִן", "לָֽמֶד-חֵֽית", "לָֽמֶד-טֵֽית", "מֵֽם", "מֵֽם-אָֽלֶף", "מֵֽם-בֵּֽית", "מֵֽם-גִּֽימֶל", "מֵֽם-דָּֽלֶת", "מֵֽם-הֵֽא", "מֵֽם-וָֽו", "מֵֽם-זַֽיִן", "מֵֽם-חֵֽית", "מֵֽם-טֵֽית", "נֽוּן", "נֽוּן-אָֽלֶף", "נֽוּן-בֵּֽית", "נֽוּן-גִּֽימֶל", "נֽוּן-דָּֽלֶת", "נֽוּן-הֵֽא", "נֽוּן-וָֽו", "נֽוּן-זַֽיִן", "נֽוּן-חֵֽית", "נֽוּן-טֵֽית", "סָֽמֶךְ"]
otiyot_gematria = [remove_nikud(ot) for ot in otiyot_gematria]


# Load the data of links_for_audio_from_929.json

with open('WhisperTimeSync/links_for_audio_from_929.json', 'r') as f:
    links_for_audio_from_929 = json.load(f)


import requests
url = links_for_audio_from_929["books"][0]["chapters"][0]["link"]
# Download (to RAM) the audio file for the first chapter of Bereshit
r = requests.get(url, allow_redirects=True)


# for book in books_data:
#     print(book['name'])
#     for chapter in range(1, book['chapters'] + 1):
#         print(chapter)
#         print(links_for_audio_from_929[book['sefaria_name']][str(chapter)])


#         להוסיף את ההורדה של האודיו ומשיכת הטקסט הרלוונטי
#         ואז להפעיל את הקוד של וויספרטיימסינק על הטקסט והאודיו







In [ ]:
!cat /content/51.Shmot_1.srt

# Synchronize

In [ ]:
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar /content/51.Shmot_1.srt /content/Exodus.1.txt he

In [ ]:
!cat /content/Exodus.1.txt.srt